# Wikipedia Redirect Index Demo

这个 notebook 用来加载、检查和测试 `WikipediaRedirectIndex`。

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from dataclasses import asdict

from pathlib import Path

from wikipedia_redirects import (
    WikipediaRedirectIndex,
    get_bucket_key,
    normalize_wikipedia_title,
)

INDEX_DIR = Path("data/wikipedia_redirects/redirect_index")
INDEX_DIR

PosixPath('data/wikipedia_redirects/redirect_index')

In [3]:
if not INDEX_DIR.exists():
    raise FileNotFoundError(f"Index directory not found: {INDEX_DIR}")

with WikipediaRedirectIndex(INDEX_DIR) as index:
    stats = index.stats()
    graph_stats = index.graph_stats()
    metadata = {
        "wiki": index.get_metadata("wiki"),
        "dump_tag": index.get_metadata("dump_tag"),
        "page_dump": index.get_metadata("page_dump"),
        "redirect_dump": index.get_metadata("redirect_dump"),
    }

{
    "stats": stats,
    "graph_stats": graph_stats,
    "metadata": metadata,
}

{'stats': {'canonical_pages': 2629590, 'redirects': 8429635},
 'graph_stats': {'redirect_nodes': 8429635,
  'target_nodes': 2629590,
  'max_total_nodes': 11059225,
  'directed_edges': 8429635,
  'undirected_edges': 8429635},
 'metadata': {'wiki': 'enwiki',
  'dump_tag': 'latest',
  'page_dump': 'data/wikipedia_redirects/raw/enwiki-latest-page.sql.gz',
  'redirect_dump': 'data/wikipedia_redirects/raw/enwiki-latest-redirect.sql.gz'}}

## 1. 单个标题查询

In [4]:
title = "USA"

normalized = normalize_wikipedia_title(title)
bucket_key = get_bucket_key(normalized)

with WikipediaRedirectIndex(INDEX_DIR) as index:
    canonical = index.resolve_redirect(title)
    synonyms = index.get_filtered_synonyms(title)

{
    "title": title,
    "normalized": normalized,
    "bucket_key": bucket_key,
    "canonical": canonical,
    "synonym_count": len(synonyms),
    "synonyms": synonyms,
}

{'title': 'USA',
 'normalized': 'usa',
 'bucket_key': 'us',
 'canonical': 'united states',
 'synonym_count': 6,
 'synonyms': ['united states',
  'american united states',
  'federal united states',
  'states united',
  'united american states',
  'united states america']}

## 2. 批量测试多个查询词

In [5]:
queries = [
    "USA",
    "U.S.A.",
    "US",
    "United States",
    "NYC",
    "New York City",
    "UK",
    "United Kingdom",
]

rows = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for query in queries:
        rows.append({
            "query": query,
            "normalized": normalize_wikipedia_title(query),
            "bucket": get_bucket_key(normalize_wikipedia_title(query)),
            "canonical": index.resolve_redirect(query),
            "synonym_count": len(index.get_synonyms(query)),
        })

rows

[{'query': 'USA',
  'normalized': 'usa',
  'bucket': 'us',
  'canonical': 'united states',
  'synonym_count': 6},
 {'query': 'U.S.A.',
  'normalized': 'u s a',
  'bucket': 'us',
  'canonical': 'united states',
  'synonym_count': 6},
 {'query': 'US',
  'normalized': 'us',
  'bucket': 'us',
  'canonical': 'chicago med season 1',
  'synonym_count': 1},
 {'query': 'United States',
  'normalized': 'united states',
  'bucket': 'un',
  'canonical': 'american wine',
  'synonym_count': 6},
 {'query': 'NYC',
  'normalized': 'nyc',
  'bucket': 'ny',
  'canonical': 'new york city',
  'synonym_count': 15},
 {'query': 'New York City',
  'normalized': 'new york city',
  'bucket': 'ne',
  'canonical': 'the men',
  'synonym_count': 15},
 {'query': 'UK',
  'normalized': 'uk',
  'bucket': 'uk',
  'canonical': 'ukca marking',
  'synonym_count': 1},
 {'query': 'United Kingdom',
  'normalized': 'united kingdom',
  'bucket': 'un',
  'canonical': 'kingdom of great britain',
  'synonym_count': 3}]

## 3. 查看某个 canonical 的同义词

In [6]:
canonical_title = "USA"

with WikipediaRedirectIndex(INDEX_DIR) as index:
    synonyms = index.get_synonyms(canonical_title)

print(f"Canonical: {canonical_title}")
print(f"Total synonyms: {len(synonyms)}")
synonyms[:100]

Canonical: USA
Total synonyms: 6


['united states',
 'american united states',
 'federal united states',
 'states united',
 'united american states',
 'united states america']

## 4. 直接查看 bucket 文件内容

In [7]:
import gzip
import pickle

bucket_to_inspect = "us"
redirect_bucket_path = INDEX_DIR / "redirect_buckets" / f"{bucket_to_inspect}.pkl.gz"
canonical_bucket_path = INDEX_DIR / "canonical_buckets" / f"{bucket_to_inspect}.pkl.gz"

def load_pickle_gz(path: Path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

redirect_bucket = load_pickle_gz(redirect_bucket_path) if redirect_bucket_path.exists() else {}
canonical_bucket = load_pickle_gz(canonical_bucket_path) if canonical_bucket_path.exists() else {}

print("redirect bucket size:", len(redirect_bucket))
print("canonical bucket size:", len(canonical_bucket))

redirect bucket size: 27449
canonical bucket size: 7557


In [8]:
list(redirect_bucket.items())[:20]

[('usstandardofliving',
  ('usstandardofliving', 'standard of living in the united states')),
 ('us', ('us', 'chicago med season 1')),
 ('us internal revenue service',
  ('us internal revenue service', 'internal revenue service')),
 ('us election 2000', ('us election 2000', '2000 united states elections')),
 ('ussr', ('ussr', 'u s s r')),
 ('u s congress representatives from guam',
  ('u s congress representatives from guam',
   "guam's at-large congressional district")),
 ('u s congress representatives from u s virgin islands',
  ('u s congress representatives from u s virgin islands',
   "united states virgin islands' at-large congressional district")),
 ('user-friendliness', ('user-friendliness', 'usability')),
 ('usa', ('usa', 'united states')),
 ('useless language', ('useless language', 'language game')),
 ('u s', ('u s', 'united states')),
 ('usemod', ('usemod', 'usemodwiki')),
 ('usama bin laden', ('usama bin laden', 'osama bin laden')),
 ('us federal reserve bank', ('us federal

In [9]:
list(canonical_bucket.items())[:10]

[('usemodwiki',
  {'title': 'usemodwiki',
   'redirects': ['atiswiki',
    'clifford adams',
    'cve-2004-1397',
    'cvwiki',
    'use mod wiki',
    'usemod',
    'usemod wiki']}),
 ('u s route 12 in michigan',
  {'title': 'u s route 12 in michigan',
   'redirects': ['auxiliary routes of u s route 12',
    'bannered routes of u s route 12',
    'm-151',
    'm-23',
    'michigan avenue',
    'michigan state highway 23',
    'old u s 12',
    'special routes of u s route 112',
    'st joseph trail',
    'u s highway 112',
    'u s highway 112s',
    'u s highway 12',
    'u s highway 12 business',
    'u s highway 12 in michigan',
    'u s route 112',
    'u s route 112 business',
    'u s route 112 bypass',
    'u s route 112 in indiana',
    'u s route 112 in michigan',
    'u s route 112s',
    'u s route 112s in indiana',
    'u s route 112s in michigan',
    'u s route 12',
    'u s route 12 alternate',
    'u s route 12 business',
    'united states highway 112',
    'united st

## 5. 遍历前几个 redirect pair

In [10]:
pairs = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for idx, pair in enumerate(index.iter_pairs()):
        pairs.append(pair)
        if idx >= 19:
            break

pairs

[RedirectPair(redirect='00 gauge', canonical='oo gauge'),
 RedirectPair(redirect='00 scale', canonical='oo gauge'),
 RedirectPair(redirect='007 in new york', canonical='octopussy and the living daylights'),
 RedirectPair(redirect='00-xx', canonical='mathematics'),
 RedirectPair(redirect='00axx', canonical='mathematics'),
 RedirectPair(redirect="00's", canonical='00s'),
 RedirectPair(redirect='0078h', canonical='dead cities red seas & lost ghosts'),
 RedirectPair(redirect='00 void', canonical='sunn o'),
 RedirectPair(redirect='007 the world is not enough', canonical='the world is not enough'),
 RedirectPair(redirect='007 golden eye', canonical='goldeneye 007'),
 RedirectPair(redirect='007 vehicles', canonical='list of james bond vehicles'),
 RedirectPair(redirect="'00s", canonical='00s'),
 RedirectPair(redirect="'00's", canonical='00s'),
 RedirectPair(redirect='007 frwl', canonical='from russia with love'),
 RedirectPair(redirect='0 0 void', canonical='sunn o'),
 RedirectPair(redirect='

## 6. 写一个便于反复调用的小函数

In [11]:
def inspect_title(title: str, synonym_limit: int = 20):
    normalized = normalize_wikipedia_title(title)
    bucket_key = get_bucket_key(normalized)
    with WikipediaRedirectIndex(INDEX_DIR) as index:
        canonical = index.resolve_redirect(title)
        synonyms = index.get_synonyms(title)
    return {
        "title": title,
        "normalized": normalized,
        "bucket_key": bucket_key,
        "canonical": canonical,
        "synonym_count": len(synonyms),
        "synonyms": synonyms[:synonym_limit],
    }

inspect_title("USA")

{'title': 'USA',
 'normalized': 'usa',
 'bucket_key': 'us',
 'canonical': 'united states',
 'synonym_count': 6,
 'synonyms': ['united states',
  'american united states',
  'federal united states',
  'states united',
  'united american states',
  'united states america']}

## 7. 查看图节点和邻居

In [12]:
node_query = "USA"

with WikipediaRedirectIndex(INDEX_DIR) as index:
    node = index.get_node(node_query)
    neighbors = index.get_neighbors(node_query)

{
    "query": node_query,
    "node": asdict(node) if node else None,
    "neighbor_count": len(neighbors),
    "neighbors": neighbors[:20],
}

{'query': 'USA',
 'node': {'node_id': 't:usa',
  'title': 'usa',
  'normalized_title': 'usa',
  'redirect_target_id': 't:united states',
  'redirect_target_title': 'united states',
  'incoming_redirects': ('air on line',
   'ansbach',
   'detzelbach',
   'hainbach',
   'schleichenbach',
   'schlichenbach',
   'u s a',
   'unconformable social amputees',
   'usa river')},
 'neighbor_count': 10,
 'neighbors': ['united states',
  'air on line',
  'ansbach',
  'detzelbach',
  'hainbach',
  'schleichenbach',
  'schlichenbach',
  'u s a',
  'unconformable social amputees',
  'usa river']}

## 8. 测试 n-hop 连通和最短路径

In [13]:
examples = [
    ("USA", "United States", 1),
    ("USA", "United States", 0),
    ("USA", "US", 3),
    ("USA", "US", 2),
    ("United States", "American wine", 1),
    ("USA", "United Kingdom", 5),
]

rows = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for source_title, target_title, max_hops in examples:
        rows.append({
            "source": source_title,
            "target": target_title,
            "max_hops": max_hops,
            "distance": index.hop_distance(source_title, target_title),
            "within_n_hops": index.are_connected_within_hops(source_title, target_title, max_hops),
            "shortest_path": index.shortest_path(source_title, target_title),
        })

rows

[{'source': 'USA',
  'target': 'United States',
  'max_hops': 1,
  'distance': 1,
  'within_n_hops': True,
  'shortest_path': ['usa', 'united states']},
 {'source': 'USA',
  'target': 'United States',
  'max_hops': 0,
  'distance': 1,
  'within_n_hops': False,
  'shortest_path': ['usa', 'united states']},
 {'source': 'USA',
  'target': 'US',
  'max_hops': 3,
  'distance': 3,
  'within_n_hops': True,
  'shortest_path': ['usa', 'united states', 'u s', 'us']},
 {'source': 'USA',
  'target': 'US',
  'max_hops': 2,
  'distance': 3,
  'within_n_hops': False,
  'shortest_path': ['usa', 'united states', 'u s', 'us']},
 {'source': 'United States',
  'target': 'American wine',
  'max_hops': 1,
  'distance': 1,
  'within_n_hops': True,
  'shortest_path': ['united states', 'american wine']},
 {'source': 'USA',
  'target': 'United Kingdom',
  'max_hops': 5,
  'distance': 12,
  'within_n_hops': False,
  'shortest_path': ['usa',
   'united states',
   'us',
   'red',
   'ncis los angeles season 4',
 

## 9. 写一个便于反复测试图关系的小函数

In [14]:
def inspect_relation(source_title: str, target_title: str, max_hops: int = 2, directed: bool = False):
    with WikipediaRedirectIndex(INDEX_DIR) as index:
        source_node = index.get_node(source_title)
        target_node = index.get_node(target_title)
        return {
            "source_node": asdict(source_node) if source_node else None,
            "target_node": asdict(target_node) if target_node else None,
            "directed": directed,
            "max_hops": max_hops,
            "distance": index.hop_distance(source_title, target_title, directed=directed),
            "within_n_hops": index.are_connected_within_hops(source_title, target_title, max_hops, directed=directed),
            "path": index.shortest_path(source_title, target_title, directed=directed),
        }

inspect_relation("USA", "U.S.A.", max_hops=2)

{'source_node': {'node_id': 't:usa',
  'title': 'usa',
  'normalized_title': 'usa',
  'redirect_target_id': 't:united states',
  'redirect_target_title': 'united states',
  'incoming_redirects': ('air on line',
   'ansbach',
   'detzelbach',
   'hainbach',
   'schleichenbach',
   'schlichenbach',
   'u s a',
   'unconformable social amputees',
   'usa river')},
 'target_node': {'node_id': 't:u s a',
  'title': 'u s a',
  'normalized_title': 'u s a',
  'redirect_target_id': 't:united states',
  'redirect_target_title': 'united states',
  'incoming_redirects': ('42nd parallel',
   'bedroom boom',
   "dos passos's u s a trilogy",
   'nineteen nineteen',
   'rivaz of red',
   'the 42nd parallel',
   'the big money',
   'u nited s tate of a tlanta',
   'u s a trilogy',
   'united state of atlanta',
   'usa',
   'usa trilogy')},
 'directed': False,
 'max_hops': 2,
 'distance': 1,
 'within_n_hops': True,
 'path': ['usa', 'u s a']}